# ETL Google Colab - Academic Capacity Analytics

Notebook ini akan otomatis membaca file CSV mentah dari Google Drive Anda, melakukan proses Extract-Transform-Load, dan menyimpan hasilnya kembali ke Google Drive Anda pada folder `Star_Schema` dan `Processed`.

**Catatan:** Saat pertama kali dijalankan, Colab akan meminta izin akses ke Google Drive Anda.

In [ ]:
# ============================================================
# 1. MOUNT GOOGLE DRIVE & IMPORT LIBRARY
# ============================================================
from google.colab import drive
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive (Akan muncul popup minta izin akses)
drive.mount('/content/drive')

# ============================================================
# 2. SETUP PATH KE GOOGLE DRIVE
# ============================================================
# Mengarah ke folder "College" di My Drive Anda
ROOT = '/content/drive/MyDrive/College'

PATH_RAW_PRODI  = os.path.join(ROOT, 'Processed', 'unsil_prodi_fresh.csv')
PATH_RAW_UNIV   = os.path.join(ROOT, 'Processed', 'unsil_univ_fresh.csv')
PATH_OUT_SCHEMA = os.path.join(ROOT, 'Star_Schema')
PATH_OUT_MASTER = os.path.join(ROOT, 'Processed', 'master_looker_unsil.csv')

# Pastikan folder output ada
os.makedirs(PATH_OUT_SCHEMA, exist_ok=True)

# Cek keberadaan file sebelum lanjut
if not os.path.exists(PATH_RAW_PRODI):
    print(f"\n❌ ERROR: File tidak ditemukan di path:\n{PATH_RAW_PRODI}")
    print("Pastikan folder 'College' letaknya tidak di dalam folder lain di My Drive Anda.")
else:
    print(f"\n✅ File ditemukan! Memulai proses ETL...\n")

    # ============================================================
    # 3. EXTRACT
    # ============================================================
    print("=" * 60)
    print("TAHAP EXTRACT")
    print("=" * 60)
    df_prodi_raw = pd.read_csv(PATH_RAW_PRODI)
    df_univ_raw  = pd.read_csv(PATH_RAW_UNIV)
    print("File prodi   :", len(df_prodi_raw), "baris")
    print("Prodi unik   :", df_prodi_raw['nama_program_studi'].nunique())

    # ============================================================
    # 4. TRANSFORM
    # ============================================================
    print("\n" + "=" * 60)
    print("TAHAP TRANSFORM")
    print("=" * 60)
    df = df_prodi_raw.copy()
    
    # [0] Filter scope
    df = df[df['nama_universitas'].str.contains('Siliwangi', case=False, na=False)].copy()
    df['kode_pt'] = '002008'
    
    # [1] Drop null kritis
    df = df.dropna(subset=['kode_prodi', 'tahun_pelaporan', 'rasio_dosen_mahasiswa'])
    
    # [2] Parsing periode
    df[['semester', 'tahun']] = df['tahun_pelaporan'].str.split(' ', n=1, expand=True)
    
    # [3] Konversi numerik
    num_cols = ['jumlah_dosen_penghitung_rasio','dosen_tetap','dosen_tidak_tetap','total_dosen','jumlah_mahasiswa']
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # [4] Parsing rasio
    def parse_rasio(s):
        try:
            if pd.isna(s): return np.nan
            parts = str(s).split(':')
            return float(parts[1]) if len(parts) == 2 else np.nan
        except:
            return np.nan
    df['nilai_rasio'] = df['rasio_dosen_mahasiswa'].apply(parse_rasio)
    
    # [5] Standarisasi metadata
    df['nama_universitas']   = 'Universitas Siliwangi'
    df['status_pt_univ']     = 'PTN'
    df['akreditasi_pt_univ'] = 'Unggul'
    df['kode_pt']            = '002008'
    
    # [6] Mapping Fakultas dan Rumpun Ilmu
    FAKULTAS_MAP = {
        'Agribisnis': ('Faperta', 'Sains'), 'Agroteknologi': ('Faperta', 'Sains'),
        'Akuntansi': ('FEB', 'Sosial'), 'Ekonomi Pembangunan': ('FEB', 'Sosial'),
        'Ekonomi Syari\'ah': ('FEB', 'Sosial'), 'Gizi': ('FIK', 'Sains'),
        'Ilmu Manajemen': ('FEB', 'Sosial'), 'Ilmu Pertanian': ('Faperta', 'Sains'),
        'Ilmu Politik': ('FISIP', 'Sosial'), 'Informatika': ('FT', 'Sains'),
        'Kesehatan Masyarakat': ('FIK', 'Sains'), 'Manajemen': ('FEB', 'Sosial'),
        'Manajemen Mutu Halal': ('FEB', 'Sosial'), 'Pendidikan': ('FKIP', 'Sosial'),
        'Pendidikan Bahasa Indonesia': ('FKIP', 'Sosial'), 'Pendidikan Bahasa Inggris': ('FKIP', 'Sosial'),
        'Pendidikan Biologi': ('FKIP', 'Sains'), 'Pendidikan Ekonomi': ('FKIP', 'Sosial'),
        'Pendidikan Fisika': ('FKIP', 'Sains'), 'Pendidikan Geografi': ('FKIP', 'Sosial'),
        'Pendidikan Ilmu Pengetahuan Alam': ('FKIP', 'Sains'), 'Pendidikan Jasmani': ('FKIP', 'Sosial'),
        'Pendidikan Kependudukan & Lingkungan Hidup': ('FKIP', 'Sosial'), 'Pendidikan Masyarakat': ('FKIP', 'Sosial'),
        'Pendidikan Matematika': ('FKIP', 'Sains'), 'Pendidikan Profesi Guru': ('FKIP', 'Sosial'),
        'Pendidikan Sejarah': ('FKIP', 'Sosial'), 'Perbankan dan Keuangan': ('FEB', 'Sosial'),
        'Perbankan dan Keuangan Digital': ('FEB', 'Sosial'), 'Sains Data': ('FT', 'Sains'),
        'Sistem Informasi': ('FT', 'Sains'), 'Teknik Elektro': ('FT', 'Sains'),
        'Teknik Sipil': ('FT', 'Sains'), 'Hukum Bisnis': ('FEB', 'Sosial'),
        'Teknologi Pangan dan Hasil Pertanian': ('Faperta', 'Sains')
    }
    df['fakultas'] = df['nama_program_studi'].apply(lambda x: FAKULTAS_MAP.get(x, ('Lainnya', 'Lainnya'))[0])
    df['rumpun_ilmu'] = df['nama_program_studi'].apply(lambda x: FAKULTAS_MAP.get(x, ('Lainnya', 'Lainnya'))[1])
    
    print("Transformasi selesai:", len(df), "baris siap diproses.")

    # ============================================================
    # 5. LOAD (Simpan ke Google Drive)
    # ============================================================
    print("\n" + "=" * 60)
    print("TAHAP LOAD KE GOOGLE DRIVE")
    print("=" * 60)
    
    dim_waktu = (df[['tahun_pelaporan','semester','tahun']].drop_duplicates().sort_values('tahun_pelaporan').reset_index(drop=True))
    dim_waktu.insert(0, 'id_waktu', dim_waktu.index + 1)
    dim_waktu['tahun'] = dim_waktu['tahun'].astype(int)
    
    dim_univ = pd.DataFrame([{'id_universitas': '002008', 'nama_universitas': 'Universitas Siliwangi', 'kota': 'Kota Tasikmalaya', 'provinsi': 'Prov. Jawa Barat', 'status_pt': 'PTN', 'akreditasi_institusi': 'Unggul'}])
    
    latest_period = df['tahun_pelaporan'].max()
    dim_prodi = (df[df['tahun_pelaporan']==latest_period][['kode_prodi','nama_program_studi','fakultas','rumpun_ilmu','jenjang','status_prodi','akreditasi_prodi']].drop_duplicates(subset=['kode_prodi']).sort_values('nama_program_studi').reset_index(drop=True).rename(columns={'kode_prodi':'id_prodi'}))
    
    fact = df.merge(dim_waktu[['id_waktu','tahun_pelaporan']], on='tahun_pelaporan', how='left')
    fact_table = fact[['kode_pt','kode_prodi','id_waktu','jumlah_dosen_penghitung_rasio','dosen_tetap','dosen_tidak_tetap','total_dosen','jumlah_mahasiswa','rasio_dosen_mahasiswa','nilai_rasio']].rename(columns={'kode_pt':'id_universitas','kode_prodi':'id_prodi'})
    fact_table = fact_table.dropna(subset=['id_universitas','id_prodi']).reset_index(drop=True)
    
    master = df[['tahun_pelaporan','semester','tahun','nama_program_studi','jenjang','status_prodi','akreditasi_prodi','nama_universitas','jumlah_mahasiswa','jumlah_dosen_penghitung_rasio','dosen_tetap','dosen_tidak_tetap','total_dosen','rasio_dosen_mahasiswa','nilai_rasio']].copy()
    master['kota'] = 'Kota Tasikmalaya'
    master['provinsi'] = 'Prov. Jawa Barat'
    master['kode_pt'] = '002008'
    master['fakultas'] = df['fakultas']
    master['rumpun_ilmu'] = df['rumpun_ilmu']
    master = master.sort_values(['tahun_pelaporan','nama_program_studi']).reset_index(drop=True)
    
    dim_waktu.to_csv(os.path.join(PATH_OUT_SCHEMA,'Dim_Waktu.csv'), index=False)
    dim_univ.to_csv(os.path.join(PATH_OUT_SCHEMA,'Dim_Universitas.csv'), index=False)
    dim_prodi.to_csv(os.path.join(PATH_OUT_SCHEMA,'Dim_Prodi.csv'), index=False)
    fact_table.to_csv(os.path.join(PATH_OUT_SCHEMA,'Fact_Kapasitas_Pendidikan.csv'), index=False)
    master.to_csv(PATH_OUT_MASTER, index=False)
    
    print("✅ BERHASIL DISIMPAN KE GOOGLE DRIVE!")
    print(f"- {PATH_OUT_MASTER}")
    print("ETL SELESAI - TIDAK ADA ERROR")
